In [9]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver


In [2]:
load_dotenv()
llm=ChatGroq(model="llama-3.3-70b-versatile")

In [3]:
class Jokestate(TypedDict):
    topic:str
    joke:str
    explanation:str


In [5]:
def generate_joke(state:Jokestate):
    prompt=f"generate a joke on the topic {state['topic']}"
    response=llm.invoke(prompt).content
    return {"joke":response}
def generate_explanation(state:Jokestate):
    prompt=f"generate a expalnation for this joke -{state['joke']}"
    response=llm.invoke(prompt)
    return {"explanation":response}

In [10]:
graph=StateGraph(Jokestate)

graph.add_node("generate_joke",generate_joke)
graph.add_node("generate_explanation",generate_explanation)

graph.add_edge(START,"generate_joke")
graph.add_edge("generate_joke","generate_explanation")
graph.add_edge("generate_explanation",END)

checkpointer=InMemorySaver()

workflow=graph.compile(checkpointer=checkpointer)

In [14]:
config2={"configurable":{"thread_id":"2"}}
workflow.invoke({"topic":"sport:cricket"},config=config2)

{'topic': 'sport:cricket',
 'joke': 'Why did the cricket ball go to therapy?\n\nBecause it was feeling a little "bowled" over by its emotions and was struggling to "catch" a break!',
 'explanation': AIMessage(content='A clever joke that combines cricket terminology with emotional struggles. Let\'s break it down:\n\nThe joke starts by setting up a unexpected scenario: a cricket ball going to therapy. This already piques the listener\'s interest, as inanimate objects don\'t typically seek therapy.\n\nThe punchline relies on a play on words, using cricket terminology to describe the ball\'s emotional state. "Bowled" has a double meaning here:\n\n1. In cricket, a ball can be bowled, meaning it\'s thrown by the bowler towards the batsman.\n2. To be "bowled over" is an idiomatic expression meaning to be overwhelmed or astonished by something.\n\nIn this joke, the ball is feeling "bowled over" by its emotions, implying that it\'s struggling to cope with its feelings and is overwhelmed.\n\nThe

In [17]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'sport:cricket', 'joke': 'Why did the cricket ball go to therapy?\n\nBecause it was feeling a little "bowled" over by its emotions and was struggling to "catch" a break!', 'explanation': AIMessage(content='A clever joke that combines cricket terminology with emotional struggles. Let\'s break it down:\n\nThe joke starts by setting up a unexpected scenario: a cricket ball going to therapy. This already piques the listener\'s interest, as inanimate objects don\'t typically seek therapy.\n\nThe punchline relies on a play on words, using cricket terminology to describe the ball\'s emotional state. "Bowled" has a double meaning here:\n\n1. In cricket, a ball can be bowled, meaning it\'s thrown by the bowler towards the batsman.\n2. To be "bowled over" is an idiomatic expression meaning to be overwhelmed or astonished by something.\n\nIn this joke, the ball is feeling "bowled over" by its emotions, implying that it\'s struggling to cope with its feelings and is 

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'cricket', 'joke': 'Why did the cricket go to the doctor?\n\nBecause it had a "sticky" wicket situation and was feeling a little "bowled" over! (get it?)', 'explanation': AIMessage(content='A classic joke that combines two different domains: cricket and medicine. Let\'s break it down:\n\nThe joke starts by setting up a familiar scenario: an animal (a cricket) going to the doctor. This is a common trope in jokes, where an unexpected character seeks medical attention.\n\nThe punchline relies on wordplay, using cricket terminology to create a humorous connection between the cricket\'s health issues and the sport of cricket. Here\'s how it works:\n\n* "Sticky wicket situation" is a phrase borrowed from cricket, where a "sticky wicket" refers to a difficult or precarious situation on the field, often caused by a wet or damp pitch. In this joke, the phrase is used to describe the cricket\'s health issues, implying that it\'s in a tricky or challenging situatio

# Time Travel

In [19]:
workflow.get_state({"configurable":{"thread_id":"1",'checkpoint_id': '1f0e2258-e9a5-6e57-8000-7c2d5bc99c10'}})

StateSnapshot(values={'topic': 'cricket'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f0e2258-e9a5-6e57-8000-7c2d5bc99c10'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2025-12-26T06:38:58.263407+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0e2258-e9a1-625f-bfff-99dee93e4f6f'}}, tasks=(PregelTask(id='069e5d81-5ebc-a779-b27d-3e08f65443bc', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the cricket go to the doctor?\n\nBecause it had a "sticky" wicket situation and was feeling a little "bowled" over! (get it?)'}),), interrupts=())

In [20]:
workflow.invoke(None,{"configurable":{"thread_id":"1",'checkpoint_id': '1f0e2258-e9a5-6e57-8000-7c2d5bc99c10'}})

{'topic': 'cricket',
 'joke': 'Why did the cricket go to the doctor?\n\nBecause it had a "sticky" wicket and was feeling a little "bowled" over! (get it?)',
 'explanation': AIMessage(content='A classic joke that combines wordplay with a dash of cricket terminology. Let\'s break it down:\n\nThe joke starts by setting up a familiar scenario: a cricket (the insect) going to the doctor. This already implies that something is amiss with the cricket\'s health.\n\nThe punchline relies on a play on words, using cricket terminology in a clever way. In cricket, a "wicket" refers to the set of three stumps and two bails that a batsman defends. A "sticky wicket" is a colloquialism that describes a difficult or challenging situation, often used to describe a tricky or precarious position on the cricket field.\n\nIn this joke, the cricket has a "sticky wicket," which is a clever double entendre. The phrase is used to describe the cricket\'s situation, but it also references the fact that crickets ar

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'cricket', 'joke': 'Why did the cricket go to the doctor?\n\nBecause it had a "sticky" wicket and was feeling a little "bowled" over! (get it?)', 'explanation': AIMessage(content='A classic joke that combines wordplay with a dash of cricket terminology. Let\'s break it down:\n\nThe joke starts by setting up a familiar scenario: a cricket (the insect) going to the doctor. This already implies that something is amiss with the cricket\'s health.\n\nThe punchline relies on a play on words, using cricket terminology in a clever way. In cricket, a "wicket" refers to the set of three stumps and two bails that a batsman defends. A "sticky wicket" is a colloquialism that describes a difficult or challenging situation, often used to describe a tricky or precarious position on the cricket field.\n\nIn this joke, the cricket has a "sticky wicket," which is a clever double entendre. The phrase is used to describe the cricket\'s situation, but it also references the f

In [24]:
workflow.invoke({"topic":"kabaddi"},{"configurable":{"thread_id":"1",'checkpoint_id': '1f0e2258-e9a5-6e57-8000-7c2d5bc99c10'}})

{'topic': 'kabaddi',
 'joke': 'Why did the kabaddi player bring a ladder to the match?\n\nBecause he wanted to take his raid to the next level.',
 'explanation': AIMessage(content='A clever play on words. Here\'s a breakdown of the joke:\n\nThe joke is a pun that relies on a double meaning of the phrase "take it to the next level." In the context of kabaddi, a raid refers to a key play where a player from one team enters the opponent\'s territory to score points. The phrase "take it to the next level" is a common idiomatic expression that means to improve or elevate something, in this case, the player\'s raid.\n\nHowever, in this joke, the phrase "take it to the next level" has a literal meaning as well. The kabaddi player brings a ladder to the match, which is a physical object that allows someone to climb to a higher level or elevation. The punchline is funny because it\'s a clever and unexpected connection between the literal meaning of the phrase (using a ladder to climb) and the i

In [25]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'kabaddi', 'joke': 'Why did the kabaddi player bring a ladder to the match?\n\nBecause he wanted to take his raid to the next level.', 'explanation': AIMessage(content='A clever play on words. Here\'s a breakdown of the joke:\n\nThe joke is a pun that relies on a double meaning of the phrase "take it to the next level." In the context of kabaddi, a raid refers to a key play where a player from one team enters the opponent\'s territory to score points. The phrase "take it to the next level" is a common idiomatic expression that means to improve or elevate something, in this case, the player\'s raid.\n\nHowever, in this joke, the phrase "take it to the next level" has a literal meaning as well. The kabaddi player brings a ladder to the match, which is a physical object that allows someone to climb to a higher level or elevation. The punchline is funny because it\'s a clever and unexpected connection between the literal meaning of the phrase (using a ladder